In [103]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [104]:
data_folder = '/content/drive/MyDrive/TrafficFlowPrediction'

In [105]:
data_file = f'{data_folder}/Scats Data October 2006.xls'

## Setup Environment

In [106]:
!pip install numpy==1.24.4 pandas==1.5.3 scikit-learn==1.3.1 xlrd

In [107]:
import numpy as np
import pandas as pd
from typing import Sequence
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## Utility

In [108]:
def read_excel(filename: str, sheet_name: str, header: int | Sequence[int]) -> pd.DataFrame:
    return pd.read_excel(filename, sheet_name=sheet_name, header=header)

# Scaling to [0-1] for RNN algos (LSTM, GRU, ...)
def scaler(x_min: float, x_max: float):
    return lambda x: (x - x_min) / (x_max - x_min)

# Rescaling scaled values to original
def rescaler(x_min: float, x_max: float):
    return lambda x: x * (x_max - x_min) + x_min

## Data Processing

In [109]:
df = read_excel(data_file, sheet_name="Data", header=1) # Read Excel data
df = df.sort_values(["NB_LATITUDE", "NB_LONGITUDE", "Date"]) # Sort by locations and date
df

,SCATS Number,Location,CD_MELWAY,NB_LATITUDE,NB_LONGITUDE,HF VicRoads Internal,VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY,Date,...,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95
62,970,WARRIGAL_RD S of HIGH STREET_RD,060 G10,-37.8676,145.09146,10503,182,5,1,2006-10-01 00:15:00,...,95,117,87,82,55,58,57,51,41,41
63,970,WARRIGAL_RD S of HIGH STREET_RD,060 G10,-37.8676,145.09146,10503,182,5,1,2006-10-02 00:15:00,...,103,107,107,88,66,54,68,58,49,29
64,970,WARRIGAL_RD S of HIGH STREET_RD,060 G10,-37.8676,145.09146,10503,182,5,1,2006-10-03 00:15:00,...,131,144,100,100,102,86,85,68,53,37
65,970,WARRIGAL_RD S of HIGH STREET_RD,060 G10,-37.8676,145.09146,10503,182,5,1,2006-10-04 00:15:00,...,147,114,131,94,84,82,90,76,50,45
66,970,WARRIGAL_RD S of HIGH STREET_RD,060 G10,-37.8676,145.09146,10503,182,5,1,2006-10-05 00:15:00,...,185,158,127,118,92,105,113,78,64,62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,4266,AUBURN_RD N of BURWOOD_RD,045 F11,0.0000,0.00000,4397,653,1,1,2006-10-27 00:15:00,...,77,52,58,55,68,55,62,62,64,42
3338,4266,AUBURN_RD N of BURWOOD_RD,045 F11,0.0000,0.00000,4397,653,1,1,2006-10-28 00:15:00,...,60,64,76,59,49,50,71,66,51,50
3339,4266,AUBURN_RD N of BURWOOD_RD,045 F11,0.0000,0.00000,4397,653,1,1,2006-10-29 00:15:00,...,79,85,63,62,58,56,54,45,35,22
3340,4266,AUBURN_RD N of BURWOOD_RD,045 F11,0.0000,0.00000,4397,653,1,1,2006-10-30 00:15:00,...,61,65,60,47,44,40,22,30,19,15


### Grouping flows

In [110]:
lags = 7 # How many past flows data are used to predict the next flow

flow_columns = [f"V{str(i).zfill(2)}" for i in range(96)]  # Creates V00 to V95
scat_grouped = df.groupby(['NB_LATITUDE', 'NB_LONGITUDE'])[flow_columns].apply(lambda x: x.values.tolist())
scat_grouped

,,0
NB_LATITUDE,NB_LONGITUDE,
-37.867600,145.091460,"[[92, 93, 90, 55, 64, 50, 47, 36, 34, 36, 34, ..."
-37.867350,145.091950,"[[37, 30, 26, 29, 14, 20, 14, 10, 10, 8, 10, 7..."
-37.867230,145.091030,"[[47, 42, 31, 34, 40, 28, 27, 18, 11, 10, 15, ..."
-37.867030,145.091590,"[[86, 83, 52, 58, 59, 44, 31, 37, 30, 24, 16, ..."
-37.861550,145.057510,"[[66, 44, 40, 37, 34, 31, 31, 34, 19, 21, 22, ..."
...,...,...
-37.782670,145.077290,"[[29, 29, 33, 25, 17, 22, 21, 14, 10, 11, 12, ..."
-37.782087,145.077826,"[[72, 57, 50, 43, 31, 37, 34, 24, 17, 23, 12, ..."
-37.781270,145.076880,"[[91, 97, 80, 87, 63, 50, 66, 60, 59, 52, 46, ..."


### Scale lat/long

In [111]:
"""Scale lat/long"""
scat_latlong = np.array(scat_grouped.index.to_list())

# We would return those scalers in case we want to inverse transform it
lat_scaler = MinMaxScaler()
long_scaler = MinMaxScaler()

lats = scat_latlong[:, 0].reshape(-1, 1) # Get the lat column and reshaped it to 1D values
longs = scat_latlong[:, 1].reshape(-1, 1) # Get the long column and reshaped it to 1D values

lats_scaled = lat_scaler.fit_transform(lats) # Fit the scaler to the lats format and scale it to [0-1]
longs_scaled = long_scaler.fit_transform(longs) # Fit the scaler to the longs format and scale it to [0-1]

scat_latlong_scaled = np.hstack((lats_scaled, longs_scaled))
scat_latlong_scaled

array([[0.00000000e+00, 9.99949069e-01],
       [6.60194995e-06, 9.99952446e-01],
       [9.77088593e-06, 9.99946106e-01],
       [1.50524459e-05, 9.99949965e-01],
       [1.59767189e-04, 9.99715091e-01],
       [1.60559423e-04, 9.99721983e-01],
       [1.67237955e-04, 9.99718730e-01],
       [1.77460415e-04, 9.99714608e-01],
       [3.29305264e-04, 9.99964920e-01],
       [3.35643136e-04, 9.99967470e-01],
       [3.41452852e-04, 9.99965472e-01],
       [4.06416039e-04, 9.99968297e-01],
       [4.13810223e-04, 9.99970916e-01],
       [4.15394691e-04, 9.99967057e-01],
       [4.20340872e-04, 9.99968957e-01],
       [5.33173478e-04, 9.99681459e-01],
       [5.37134648e-04, 9.99684560e-01],
       [5.40303584e-04, 9.99677461e-01],
       [5.47697768e-04, 9.99629425e-01],
       [5.48490002e-04, 9.99682286e-01],
       [5.61957980e-04, 9.99620466e-01],
       [7.81934952e-04, 9.99985941e-01],
       [7.87744668e-04, 9.99740177e-01],
       [7.93290306e-04, 9.99736180e-01],
       [7.935543

### Scale flow & Flow window slicing

In [112]:
"""Scale flow"""
scat_data = scat_grouped.values

flow_max = np.array(scat_data.max()).max()
flow_min = np.array(scat_data.min()).min()

# We would return scaler and rescaler in case we want to scale and inverse scale it
flow_scaler = scaler(flow_min, flow_max) # Get the scaler of flow data
flow_rescaler = rescaler(flow_min, flow_max) # Get the rescaler of flow data

X_latlong = []
X_flow = []
y = []

for i, flow in enumerate(scat_grouped.values):
    flow = np.array(flow, dtype=float).flatten()
    flow = np.vectorize(flow_scaler)(flow)

    """Flow window slicing"""
    indices = np.arange(lags, len(flow))
    offset = np.arange(-lags, 0)

    for idx in indices:
        past_flow = flow[idx + offset]  # past lags
        target = flow[idx]              # current target
        X_flow.append(past_flow)
        X_latlong.append(scat_latlong_scaled[i])
        y.append(target)

### Split training/test data

In [113]:
# Convert lists to numpy arrays
X_latlong = np.array(X_latlong)
X_flow = np.array(X_flow)
y = np.array(y)

# Combine into a single X tuple for train/test split
combined_X = list(zip(X_latlong, X_flow))

# Perform train-test split
X_train_comb, X_test_comb, y_train, y_test = train_test_split(combined_X, y, random_state=0, train_size=0.75)

# Unzip the combined tuples back into separate arrays
X_latlong_train, X_flow_train = zip(*X_train_comb)
X_latlong_test, X_flow_test = zip(*X_test_comb)

# Convert back to np.ndarray
X_latlong_train = np.array(X_latlong_train)
X_flow_train = np.array(X_flow_train)
X_latlong_test = np.array(X_latlong_test)
X_flow_test = np.array(X_flow_test)

In [114]:
X_latlong_train

array([[1.12021887e-03, 9.99615159e-01],
       [1.15428493e-03, 9.99553684e-01],
       [1.11784217e-03, 9.99853341e-01],
       ...,
       [1.11388100e-03, 9.99618260e-01],
       [9.43814765e-04, 9.99640452e-01],
       [1.55093008e-03, 9.99511230e-01]])

In [115]:
X_flow_train

array([[0.21698113, 0.25943396, 0.22484277],
       [0.21069182, 0.22012579, 0.22012579],
       [0.27672956, 0.28616352, 0.25471698],
       ...,
       [0.06603774, 0.0581761 , 0.11477987],
       [0.09748428, 0.10220126, 0.11163522],
       [0.10849057, 0.13522013, 0.13679245]])

In [116]:
y_train

array([0.25786164, 0.20440252, 0.22169811, ..., 0.14308176, 0.10849057,
       0.10534591])

In [117]:
X_latlong_test

array([[3.41452852e-04, 9.99965472e-01],
       [7.97251476e-04, 9.99982150e-01],
       [1.15428493e-03, 9.99553684e-01],
       ...,
       [1.17435486e-03, 9.99495516e-01],
       [1.38640949e-03, 9.99498480e-01],
       [1.50524459e-05, 9.99949965e-01]])

In [118]:
X_flow_test

array([[0.02044025, 0.01886792, 0.01886792],
       [0.30503145, 0.28459119, 0.18710692],
       [0.05503145, 0.04716981, 0.03773585],
       ...,
       [0.10062893, 0.1163522 , 0.08962264],
       [0.15251572, 0.16037736, 0.16981132],
       [0.43710692, 0.37264151, 0.38522013]])

In [119]:
y_test

array([0.02515723, 0.17295597, 0.06132075, ..., 0.07232704, 0.16037736,
       0.32389937])